# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset on rangeland management practices in Northern Kenya using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is described by a [Croissant schema](https://mlcommons.org/croissant/) available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"ID: {metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We inspect the available Record Sets, list their IDs, and then enumerate the fields (columns) in each Record Set.

All references occur using the `@id` of each entity, as recommended by the Croissant specification.

In [ ]:
# List available record sets and their fields using @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s) in the dataset.\n")

record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    record_set_ids.append(rs.id)
    # List fields by @id
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields (by @id):")
        for f in rs.fields:
            print(f"    - {f.id}")
    elif hasattr(rs, 'columns') and rs.columns:
        print("  Columns (by @id):")
        for c in rs.columns:
            print(f"    - {c.id}")
    else:
        print("  No fields or columns in this record set.")
    print()

## 3. Data Extraction
Load data from each specific record set into a pandas DataFrame for further analysis.

We use the record set and field `@id`s collected above.

In [ ]:
# Extract data from each record set into DataFrames using their @id
import warnings
warnings.filterwarnings('ignore')

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {list(df.columns)}")
        print(df.head(2))
    else:
        print("  No records found for this record set.")
    print()

# For demonstration, pick the first available record set with data
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break
if main_record_set_id:
    print(f"Using main record set for EDA: {main_record_set_id}")
    print(f"Available columns: {list(dataframes[main_record_set_id].columns)}")
    display(dataframes[main_record_set_id].head())
else:
    print("No populated record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records based on criteria, normalizing numeric fields, and grouping data. Entities are always referenced by their `@id` fields.

*Note: Adjust the code as needed to match the actual column names present in your record sets. For demonstration, we attempt the steps if suitable numeric and grouping columns are present.*

In [ ]:
# EDA: filter, normalize, group on numeric and categorical fields by @id
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Heuristically choose a numeric field and a group/categorical field
    numeric_field_id = None
    group_field_id = None
    
    # Search for candidate numeric field by dtype
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Search for candidate group field by unique values
    for col in df.columns:
        if df[col].nunique() < 15 and col != numeric_field_id:
            group_field_id = col
            break
    
    if numeric_field_id:
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (mean):")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA. Please verify record set structure.")
else:
    print("No main record set available for EDA.")

## 5. Visualization
Visualize data distributions or field relationships in the dataset using matplotlib or seaborn.

*If there is a numeric field and group field, provide a boxplot of the numeric field grouped by the categorical/group field. Otherwise, plot a histogram of the first available numeric column.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field
if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    if group_field_id:
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
    else:
        sns.histplot(df[numeric_field_id].dropna(), bins=20)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("Visualization not possible: no suitable numeric field available.")

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to discover record sets, load and examine data, and apply some exploratory data analysis using the dataset entities referenced by their `@id`.

Key points:
- The FAIR² dataset contains outputs from ordered logistic regression analyzing knowledge adoption in rangeland management interventions in Northern Kenya.
- All entities—record sets, fields, and columns—are referenced by `@id`s, ensuring unique and consistent access across the Croissant schema.
- Example operations included filtering and normalization of a numeric field, grouping by a categorical variable, and simple visualizations.

**Please explore additional record sets, fields, and relationships using their respective `@id`s for deeper insights!**